# LeetCode 1461 — Check If a String Contains All Binary Codes of Size K bằng Python

Link đề: https://leetcode.com/problems/check-if-a-string-contains-all-binary-codes-of-size-k/ · Độ khó: **Medium** · Chủ đề: Hash Table, String, Bit Manipulation, Rolling Hash, Hash Function

Skeleton LeetCode:

```python
class Solution:
    def hasAllCodes(self, s: str, k: int) -> bool:
        ...
```

## 1. Mục tiêu học

1. Hiểu **"mã nhị phân độ dài k"** là gì và vì sao có đúng `2^k` mã như vậy.
2. Đổi góc nhìn từ "với mỗi mã, tìm nó trong `s`" (chậm) sang "với mỗi **cửa sổ độ dài k** của `s`, ghi lại mã của nó" (nhanh).
3. Dùng **set** để đếm số chuỗi con phân biệt, và dùng **nguyên lý chuồng bồ câu** để trả `False` sớm khi `len(s) - k + 1 < 2^k`.
4. Cài đặt **bitmask cửa sổ trượt** (một dạng rolling hash) với `mask = ((mask << 1) | bit) & (2^k − 1)` để đạt `O(n)` thời gian, không tạo chuỗi con.
5. Tránh các lỗi kinh điển: thiếu cửa sổ cuối, quên cắt mask, `bin(i)` thiếu số 0 đầu, viết `2^k` (XOR!) thay cho `2**k`.

## 2. Đề bài & ví dụ

Cho một chuỗi nhị phân `s` (chỉ gồm `'0'` và `'1'`) và số nguyên `k`. Trả về `True` nếu **mọi** chuỗi nhị phân độ dài `k` đều xuất hiện như một **chuỗi con liên tiếp** (substring) của `s`; ngược lại trả `False`.

**Ví dụ 1:** `s = "00110110", k = 2` → `True`. Các mã độ dài 2 là `"00"`, `"01"`, `"10"`, `"11"`; chúng xuất hiện lần lượt ở chỉ số 0, 1, 3, 2.

**Ví dụ 2:** `s = "0110", k = 1` → `True`. Cả `"0"` và `"1"` đều có.

**Ví dụ 3:** `s = "0110", k = 2` → `False`. Mã `"00"` không xuất hiện.

**Ràng buộc:**
- `1 <= s.length <= 5 * 10^5`
- `s[i]` là `'0'` hoặc `'1'`
- `1 <= k <= 20`

Lưu ý: `k` có thể **lớn hơn** `len(s)` (ví dụ `s = "0"`, `k = 3`) — khi đó chắc chắn `False`.

In [1]:
import random
import time
from itertools import product
from typing import List

examples = [
    ("00110110", 2, True),
    ("0110", 1, True),
    ("0110", 2, False),
]
for s, k, want in examples:
    print(f"s={s!r}, k={k} -> kỳ vọng {want}")

s='00110110', k=2 -> kỳ vọng True
s='0110', k=1 -> kỳ vọng True
s='0110', k=2 -> kỳ vọng False


## 3. Kiến thức nền

### 3.1 Cắt chuỗi (slicing) và các cửa sổ độ dài k

`s[i:i+k]` lấy `k` ký tự bắt đầu từ chỉ số `i`. Chuỗi độ dài `n` có đúng `n - k + 1` cửa sổ độ dài `k` (với `i = 0, 1, ..., n-k`). Nếu `k > n` thì không có cửa sổ nào.

In [2]:
s = "00110110"
k = 2
n = len(s)
print("số cửa sổ =", n - k + 1)
for i in range(n - k + 1):
    print(i, s[i:i + k])
print("k > n:", list(range(len("01") - 5 + 1)))   # range rỗng -> không có cửa sổ

số cửa sổ = 7
0 00
1 01
2 11
3 10
4 01
5 11
6 10
k > n: []


### 3.2 `set` — tập hợp các phần tử phân biệt

`set` tự loại trùng, thêm và kiểm tra thành viên trung bình `O(1)` (băm). Ở bài này ta bỏ mọi cửa sổ vào set rồi hỏi: set có đủ `2^k` phần tử không?

In [3]:
seen = set()
for x in ["00", "01", "11", "10", "01", "11", "10"]:
    seen.add(x)
print(seen, "| số phần tử phân biệt:", len(seen))
print("'00' in seen?", "00" in seen, "| '000' in seen?", "000" in seen)

# set comprehension viết gọn:
print({s[i:i + 2] for i in range(len(s) - 1)})

{'11', '00', '10', '01'} | số phần tử phân biệt: 4
'00' in seen? True | '000' in seen? False
{'11', '00', '10', '01'}


### 3.3 Sinh mọi mã nhị phân độ dài k

Có hai cách tiện trong Python:
- `itertools.product("01", repeat=k)` sinh mọi bộ `k` ký tự lấy từ `"01"`.
- `format(i, f"0{k}b")` đổi số `i` sang nhị phân, **đệm số 0 bên trái** cho đủ `k` chữ số (khác với `bin(i)` không đệm và có tiền tố `0b`).

In [4]:
k = 3
codes1 = ["".join(t) for t in product("01", repeat=k)]
codes2 = [format(i, f"0{k}b") for i in range(2 ** k)]
print(codes1)
print(codes2)
print("giống nhau?", codes1 == codes2, "| số mã =", len(codes1), "= 2**k =", 2 ** k)
print("bin(1) =", bin(1), "-> bin thiếu số 0 đầu, có tiền tố 0b")

['000', '001', '010', '011', '100', '101', '110', '111']
['000', '001', '010', '011', '100', '101', '110', '111']
giống nhau? True | số mã = 8 = 2**k = 8
bin(1) = 0b1 -> bin thiếu số 0 đầu, có tiền tố 0b


### 3.4 Phép toán bit: dịch trái, OR, AND với mặt nạ

Coi mỗi mã `k` bit là một **số nguyên** trong `[0, 2^k − 1]`:
- `x << 1`: dịch trái 1 bit (nhân 2), chỗ trống bên phải là `0`.
- `x | b`: bật bit thấp nhất thành `b` (khi bit đó đang là 0).
- `x & ((1 << k) - 1)`: giữ lại đúng `k` bit thấp, **xoá bit tràn** bên trái.
- `int("0110", 2)` đổi chuỗi nhị phân thành số.

**Chú ý:** trong Python `2 ^ k` là **XOR**, không phải lũy thừa! Lũy thừa là `2 ** k` hoặc `1 << k`.

In [5]:
k = 3
full = (1 << k) - 1
print("mặt nạ k=3:", full, format(full, "b"))
x = int("101", 2)
print("x =", x, format(x, "03b"))
y = ((x << 1) | 1) & full          # đẩy thêm bit 1 vào bên phải, bỏ bit trái nhất
print("sau khi trượt thêm '1':", y, format(y, "03b"))
print("2 ^ 3 =", 2 ^ 3, "  nhưng 2 ** 3 =", 2 ** 3, " và 1 << 3 =", 1 << 3)

mặt nạ k=3: 7 111
x = 5 101
sau khi trượt thêm '1': 3 011
2 ^ 3 = 1   nhưng 2 ** 3 = 8  và 1 << 3 = 8


### 3.5 `all()` và dừng sớm

`all(điều_kiện for x in ...)` trả `True` nếu mọi phần tử thỏa, và **dừng ngay** ở phần tử đầu tiên sai — tiện cho cách brute force "mọi mã đều có trong s".

In [6]:
calls = []
def check(x):
    calls.append(x)
    return x < 3
print(all(check(x) for x in range(10)), "| đã gọi check với:", calls)

False | đã gọi check với: [0, 1, 2, 3]


## 4. Ý tưởng nền tảng / trực giác

Tính tay với `s = "00110110"`, `k = 2`. Liệt kê mọi cửa sổ:

| i | cửa sổ `s[i:i+2]` | mã mới? | tập đã thấy |
|---|---|---|---|
| 0 | `00` | ✔ | {00} |
| 1 | `01` | ✔ | {00, 01} |
| 2 | `11` | ✔ | {00, 01, 11} |
| 3 | `10` | ✔ | {00, 01, 11, 10} → đủ 4 = 2² |
| 4 | `01` | – | |
| 5 | `11` | – | |
| 6 | `10` | – | |

Đủ 4 mã ⇒ `True`. Với `s = "0110"`, `k = 2`: cửa sổ là `01, 11, 10` — chỉ 3 cửa sổ, **không thể** phủ 4 mã ⇒ `False` ngay mà không cần xem nội dung.

Hai quan sát:
1. Chỉ cần xét `n − k + 1` cửa sổ, không cần tìm từng mã.
2. Nếu số cửa sổ `< 2^k` thì chắc chắn thiếu (chuồng bồ câu).

In [7]:
def windows(s, k):
    return [s[i:i + k] for i in range(len(s) - k + 1)]

for s, k, _ in examples:
    w = windows(s, k)
    print(f"s={s!r}, k={k}: cửa sổ={w}, phân biệt={len(set(w))}, cần={2 ** k}")

s='00110110', k=2: cửa sổ=['00', '01', '11', '10', '01', '11', '10'], phân biệt=4, cần=4
s='0110', k=1: cửa sổ=['0', '1', '1', '0'], phân biệt=2, cần=2
s='0110', k=2: cửa sổ=['01', '11', '10'], phân biệt=3, cần=4


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** sinh lần lượt cả `2^k` mã, với mỗi mã kiểm tra `code in s` (Python tìm chuỗi con). Nếu có mã nào không tìm thấy ⇒ `False`.

Đây là cách "dịch thẳng đề bài" sang code — dễ viết, dễ tin là đúng, dùng làm **chuẩn đối chiếu** cho các cách nhanh hơn.

In [8]:
def has_all_codes_brute(s: str, k: int) -> bool:
    for i in range(2 ** k):
        code_ = format(i, f"0{k}b")
        if code_ not in s:          # tìm chuỗi con: O(n*k) xấu nhất
            return False
    return True

for s, k, want in examples:
    got = has_all_codes_brute(s, k)
    print(f"{s!r}, k={k} -> {got} (kỳ vọng {want})", "✔" if got == want else "✘")

'00110110', k=2 -> True (kỳ vọng True) ✔
'0110', k=1 -> True (kỳ vọng True) ✔
'0110', k=2 -> False (kỳ vọng False) ✔


**Độ phức tạp:** có `2^k` mã, mỗi lần `in` tốn tới `O(n·k)` (thực tế Python làm khá nhanh nhưng vẫn tỉ lệ với `n`). Tổng `O(2^k · n · k)` xấu nhất. Với `k = 20`, `n = 5·10^5`: `10^6 × 5·10^5` — quá chậm.

**Nút thắt:** mỗi mã lại quét cả chuỗi `s` từ đầu. Ta đang lặp lại công việc: chuỗi `s` chỉ có `n − k + 1` cửa sổ, nhưng ta hỏi `2^k` câu hỏi, mỗi câu quét hết các cửa sổ đó. Cell dưới đo thời gian để thấy rõ khi `k` tăng.

In [9]:
# Trường hợp xấu cho brute force: s chứa MỌI mã (ghép tất cả mã lại), nên không dừng sớm được
for k in (8, 10, 12, 13):
    s_all = "".join(format(i, f"0{k}b") for i in reversed(range(2 ** k)))
    t = time.perf_counter()
    r = has_all_codes_brute(s_all, k)
    print(f"k={k:2d}: n={len(s_all):6d}, kết quả={r}, thời gian={time.perf_counter() - t:.3f}s")

k= 8: n=  2048, kết quả=True, thời gian=0.001s
k=10: n= 10240, kết quả=True, thời gian=0.008s
k=12: n= 49152, kết quả=True, thời gian=0.104s


k=13: n=106496, kết quả=True, thời gian=0.400s


## 6. Ý tưởng thuật toán / bước đột phá

**Đảo câu hỏi:** thay vì "mỗi mã có nằm trong `s` không?", hỏi "`s` chứa **những** mã nào?". Mỗi cửa sổ độ dài `k` của `s` *chính là* một mã. Gom tất cả vào một tập rồi đếm.

**Quy tắc:**
- Nếu `len(s) − k + 1 < 2^k` **thì** trả `False` ngay (không đủ cửa sổ để phủ mọi mã).
- Nếu số cửa sổ **phân biệt** bằng `2^k` **thì** `True` (vì mọi cửa sổ đều là một mã hợp lệ độ dài `k`, mà chỉ có `2^k` mã ⇒ phân biệt đủ `2^k` nghĩa là có mặt tất cả).

**Tăng tốc tiếp:** tạo chuỗi con `s[i:i+k]` tốn `O(k)` mỗi lần. Ta coi cửa sổ là **số k bit** và trượt nó: khi sang phải một ô, bỏ bit trái nhất, thêm bit mới bên phải:

```
mask_mới = ((mask_cũ << 1) | bit_mới) & (2^k − 1)
```

Mỗi bước `O(1)` ⇒ cả thuật toán `O(n)`. Đây là dạng **rolling hash hoàn hảo** (không va chạm), vì mã k bit ↔ số nguyên duy nhất.

**Invariant của cửa sổ trượt:** sau khi xử lý ký tự `s[i]` (với `i ≥ k−1`), `mask` bằng đúng giá trị nhị phân của `s[i−k+1 .. i]`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Set các chuỗi con độ dài k

**Từng đoạn:**
1. Kiểm tra nhanh chuồng bồ câu.
2. Duyệt mọi `i` từ `0` tới `n − k`, thêm `s[i:i+k]` vào set.
3. So `len(set)` với `2**k`.

Thời gian `O(n·k)` (mỗi lần cắt chuỗi `O(k)` + băm `O(k)`), bộ nhớ `O(2^k · k)` cho các chuỗi trong set. Với `k ≤ 20` vẫn chạy tốt trên LeetCode.

In [10]:
def has_all_codes_set(s: str, k: int) -> bool:
    need = 1 << k
    if len(s) - k + 1 < need:          # chuồng bồ câu (cũng xử lý k > len(s))
        return False
    seen = set()
    for i in range(len(s) - k + 1):
        seen.add(s[i:i + k])
    return len(seen) == need

for s, k, want in examples:
    print(s, k, has_all_codes_set(s, k), want)

00110110 2 True True
0110 1 True True
0110 2 False False


**Code nộp LeetCode (Cách 2):**

In [11]:
class Solution:
    def hasAllCodes(self, s: str, k: int) -> bool:
        need = 1 << k
        if len(s) - k + 1 < need:
            return False
        return len({s[i:i + k] for i in range(len(s) - k + 1)}) == need

print([Solution().hasAllCodes(s, k) for s, k, _ in examples])

[True, True, False]


### 7.2 Cách 3 — Set số nguyên + dừng sớm

Biến thể nhỏ của Cách 2: đổi mỗi cửa sổ thành số bằng `int(s[i:i+k], 2)`, và **dừng sớm** ngay khi đã đủ `2^k` mã (không cần duyệt hết `s`). Vẫn `O(n·k)` xấu nhất, nhưng thường nhanh hơn khi chuỗi ngẫu nhiên đủ mã sớm. Ý nghĩa chính: chuẩn bị tư duy "mã = số nguyên" cho Cách 4.

In [12]:
def has_all_codes_int(s: str, k: int) -> bool:
    need = 1 << k
    if len(s) - k + 1 < need:
        return False
    seen = set()
    for i in range(len(s) - k + 1):
        seen.add(int(s[i:i + k], 2))
        if len(seen) == need:          # dừng sớm
            return True
    return False

for s, k, want in examples:
    print(s, k, has_all_codes_int(s, k), want)

00110110 2 True True
0110 1 True True
0110 2 False False


**Code nộp LeetCode (Cách 3):**

In [13]:
class Solution:
    def hasAllCodes(self, s: str, k: int) -> bool:
        need = 1 << k
        if len(s) - k + 1 < need:
            return False
        seen = set()
        for i in range(len(s) - k + 1):
            seen.add(int(s[i:i + k], 2))
            if len(seen) == need:
                return True
        return False

print([Solution().hasAllCodes(s, k) for s, k, _ in examples])

[True, True, False]


### 7.3 Cách 4 — Bitmask cửa sổ trượt (rolling hash) `O(n)`

**Từng đoạn:**
1. `full = (1 << k) - 1` là mặt nạ `k` bit toàn `1`.
2. Duyệt từng ký tự `c` ở vị trí `i`: `mask = ((mask << 1) | (c == "1")) & full`.
3. Chỉ khi `i >= k - 1` thì cửa sổ mới đủ `k` ký tự ⇒ ghi nhận `mask`.
4. Dùng mảng đánh dấu `seen = bytearray(2^k)` (nhanh và gọn hơn set), cùng biến đếm `cnt` số mã mới; `cnt == 2^k` ⇒ `True` ngay.

Thời gian `O(n)`, bộ nhớ `O(2^k)` byte (tối đa ~1 MB khi `k = 20`).

In [14]:
def has_all_codes_bitmask(s: str, k: int) -> bool:
    need = 1 << k
    if len(s) - k + 1 < need:
        return False
    full = need - 1
    seen = bytearray(need)
    mask = cnt = 0
    for i, c in enumerate(s):
        mask = ((mask << 1) | (c == "1")) & full
        if i >= k - 1 and not seen[mask]:
            seen[mask] = 1
            cnt += 1
            if cnt == need:
                return True
    return False

for s, k, want in examples:
    print(s, k, has_all_codes_bitmask(s, k), want)

00110110 2 True True
0110 1 True True
0110 2 False False


`(c == "1")` là `True/False`, mà trong Python `True == 1`, `False == 0`, nên `mask << 1 | True` cho bit 1. Code nộp LeetCode:

In [15]:
class Solution:
    def hasAllCodes(self, s: str, k: int) -> bool:
        need = 1 << k
        if len(s) - k + 1 < need:
            return False
        full = need - 1
        seen = bytearray(need)
        mask = cnt = 0
        for i, c in enumerate(s):
            mask = ((mask << 1) | (c == "1")) & full
            if i >= k - 1 and not seen[mask]:
                seen[mask] = 1
                cnt += 1
                if cnt == need:
                    return True
        return False

print([Solution().hasAllCodes(s, k) for s, k, _ in examples])

[True, True, False]


So sánh tốc độ các cách trên cùng một chuỗi ngẫu nhiên lớn (Cách 1 chỉ chạy với `k` nhỏ):

In [16]:
rng = random.Random(7)
s_big = "".join(rng.choice("01") for _ in range(200000))
for name, f in [("set chuỗi", has_all_codes_set), ("set số + dừng sớm", has_all_codes_int),
                ("bitmask", has_all_codes_bitmask)]:
    for k in (10, 16):
        t = time.perf_counter()
        r = f(s_big, k)
        print(f"{name:18s} k={k}: {r!s:5s} {time.perf_counter() - t:.3f}s")

set chuỗi          k=10: True  0.022s
set chuỗi          k=16: False 0.037s
set số + dừng sớm  k=10: True  0.001s


set số + dừng sớm  k=16: False 0.047s
bitmask            k=10: True  0.001s


bitmask            k=16: False 0.033s


## 8. TRACE từng bước

Trace Cách 4 với `s = "00110110"`, `k = 2` (`full = 0b11 = 3`):

| i | c | mask (nhị phân) | đủ k ký tự? | mã mới? | cnt |
|---|---|---|---|---|---|
| 0 | 0 | `00` (thực ra mới 1 ký tự) | ✘ | – | 0 |
| 1 | 0 | `00` | ✔ | ✔ 00 | 1 |
| 2 | 1 | `01` | ✔ | ✔ 01 | 2 |
| 3 | 1 | `11` | ✔ | ✔ 11 | 3 |
| 4 | 0 | `10` | ✔ | ✔ 10 | 4 → đủ, `True` |

Hàm `trace_bitmask` in đúng các biến này qua từng vòng lặp.

In [17]:
def trace_bitmask(s, k):
    need = 1 << k
    full = need - 1
    print(f"s={s!r}, k={k}, need={need}, full={format(full, 'b')}")
    if len(s) - k + 1 < need:
        print("  số cửa sổ", len(s) - k + 1, "< need -> False ngay")
        return False
    seen = bytearray(need)
    mask = cnt = 0
    for i, c in enumerate(s):
        old = mask
        mask = ((mask << 1) | (c == "1")) & full
        ready = i >= k - 1
        new = ready and not seen[mask]
        if new:
            seen[mask] = 1
            cnt += 1
        print(f"  i={i} c={c} mask: {format(old, f'0{k}b')} -> {format(mask, f'0{k}b')} "
              f"đủ={ready!s:5s} mới={new!s:5s} cnt={cnt}")
        if cnt == need:
            print("  đủ mọi mã -> True")
            return True
    print("  hết chuỗi, cnt =", cnt, "< need -> False")
    return False

trace_bitmask("00110110", 2)

s='00110110', k=2, need=4, full=11
  i=0 c=0 mask: 00 -> 00 đủ=False mới=False cnt=0
  i=1 c=0 mask: 00 -> 00 đủ=True  mới=True  cnt=1
  i=2 c=1 mask: 00 -> 01 đủ=True  mới=True  cnt=2
  i=3 c=1 mask: 01 -> 11 đủ=True  mới=True  cnt=3
  i=4 c=0 mask: 11 -> 10 đủ=True  mới=True  cnt=4
  đủ mọi mã -> True


True

In [18]:
trace_bitmask("0110", 2)
print()
trace_bitmask("0001011100", 3)

s='0110', k=2, need=4, full=11
  số cửa sổ 3 < need -> False ngay

s='0001011100', k=3, need=8, full=111
  i=0 c=0 mask: 000 -> 000 đủ=False mới=False cnt=0
  i=1 c=0 mask: 000 -> 000 đủ=False mới=False cnt=0
  i=2 c=0 mask: 000 -> 000 đủ=True  mới=True  cnt=1
  i=3 c=1 mask: 000 -> 001 đủ=True  mới=True  cnt=2
  i=4 c=0 mask: 001 -> 010 đủ=True  mới=True  cnt=3
  i=5 c=1 mask: 010 -> 101 đủ=True  mới=True  cnt=4
  i=6 c=1 mask: 101 -> 011 đủ=True  mới=True  cnt=5
  i=7 c=1 mask: 011 -> 111 đủ=True  mới=True  cnt=6
  i=8 c=0 mask: 111 -> 110 đủ=True  mới=True  cnt=7
  i=9 c=0 mask: 110 -> 100 đủ=True  mới=True  cnt=8
  đủ mọi mã -> True


True

## 9. Code nộp LeetCode

Bản khuyên dùng: bitmask cửa sổ trượt, `O(n)` thời gian, `O(2^k)` bộ nhớ, đúng chữ ký `hasAllCodes(self, s: str, k: int) -> bool`.

In [19]:
class Solution:
    def hasAllCodes(self, s: str, k: int) -> bool:
        need = 1 << k
        if len(s) - k + 1 < need:
            return False
        full = need - 1
        seen = bytearray(need)
        mask = cnt = 0
        for i, c in enumerate(s):
            mask = ((mask << 1) | (c == "1")) & full
            if i >= k - 1 and not seen[mask]:
                seen[mask] = 1
                cnt += 1
                if cnt == need:
                    return True
        return False

for s, k, want in examples:
    assert Solution().hasAllCodes(s, k) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

1. **Mỗi cửa sổ là một mã hợp lệ.** Cửa sổ `s[i:i+k]` có đúng `k` ký tự `0/1` nên là một trong `2^k` mã. Ngược lại, một mã xuất hiện trong `s` ⇔ nó bằng ít nhất một cửa sổ. Vậy tập mã có trong `s` **chính là** tập các cửa sổ phân biệt.
2. **Đếm đủ ⇔ có tất cả.** Tập cửa sổ phân biệt là tập con của `2^k` mã; tập con có `2^k` phần tử thì bằng cả tập. Nên `len(seen) == 2^k` ⇔ mọi mã đều xuất hiện.
3. **Chuồng bồ câu.** Có `n − k + 1` cửa sổ thì có tối đa `n − k + 1` mã phân biệt; nếu số này `< 2^k` thì chắc chắn thiếu ⇒ `False` đúng.
4. **Bitmask đúng (invariant).** Quy nạp: sau bước `i`, `mask` = giá trị của `min(i+1, k)` ký tự cuối. Bước `i+1`: dịch trái thêm bit mới, `& full` bỏ bit thứ `k+1` từ phải ⇒ còn đúng `k` ký tự cuối. Ánh xạ "chuỗi k bit → số" là song ánh nên đếm số khác nhau = đếm chuỗi khác nhau.
5. **Dừng sớm an toàn.** Khi `cnt == 2^k` đã có đủ mã; xem thêm cũng không làm mất mã nào.

## 11. Phân tích Time / Space

Gọi `n = len(s)`, `K = 2^k` (≤ 1 048 576).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Sinh mã + `in` | `O(K · n · k)` xấu nhất | `O(k)` | chỉ dùng khi `k` rất nhỏ |
| 2. Set chuỗi con | `O(n · k)` | `O(min(n, K) · k)` | gọn, dễ viết |
| 3. Set số + dừng sớm | `O(n · k)` xấu nhất | `O(min(n, K))` | thường nhanh hơn khi đủ mã sớm |
| 4. Bitmask trượt | `O(n)` | `O(K)` byte | tối ưu; không tạo chuỗi con |

Bước kiểm tra chuồng bồ câu là `O(1)` và loại ngay nhiều trường hợp (ví dụ `k = 20` mà `n < 2^20 + 19`). Trường hợp xấu nhất thực tế: `n = 5·10^5`, `k ≤ 18` (vì `2^19 > 5·10^5 − 18`, nên `k ≥ 19` luôn trả `False` ngay).

In [20]:
n = 5 * 10**5
for k in range(16, 21):
    print(f"k={k}: số cửa sổ={n - k + 1}, 2^k={1 << k}, chuồng bồ câu loại ngay? {n - k + 1 < (1 << k)}")

k=16: số cửa sổ=499985, 2^k=65536, chuồng bồ câu loại ngay? False
k=17: số cửa sổ=499984, 2^k=131072, chuồng bồ câu loại ngay? False
k=18: số cửa sổ=499983, 2^k=262144, chuồng bồ câu loại ngay? False
k=19: số cửa sổ=499982, 2^k=524288, chuồng bồ câu loại ngay? True
k=20: số cửa sổ=499981, 2^k=1048576, chuồng bồ câu loại ngay? True


## 12. Bộ test

Gồm: ví dụ đề, edge case (`k > n`, `k == n`, chuỗi toàn 0, chuỗi De Bruijn — ngắn nhất mà vẫn đủ mã), đối chiếu ngẫu nhiên giữa 4 cách (seed cố định), và test hiệu năng ở ràng buộc lớn.

In [21]:
ALL = [has_all_codes_brute, has_all_codes_set, has_all_codes_int, has_all_codes_bitmask,
       lambda s, k: Solution().hasAllCodes(s, k)]

def check_all(s, k, want):
    for f in ALL:
        got = f(s, k)
        assert got == want, (s, k, got, want)

for s, k, want in examples:
    check_all(s, k, want)

edge = [
    ("0", 1, False),            # thiếu "1"
    ("01", 1, True),
    ("0", 3, False),            # k > n
    ("0110", 4, False),         # k == n, chỉ 1 cửa sổ
    ("0000000000", 1, False),
    ("00110", 2, True),         # De Bruijn bậc 2: dài 2^2 + 2 - 1 = 5
    ("0001011100", 3, True),    # chứa De Bruijn bậc 3 "0001011100"
    ("0001011101", 3, False),   # thiếu "100"? kiểm tra bằng brute
]
for s, k, want in edge:
    if s == "0001011101":
        want = has_all_codes_brute(s, k)
    check_all(s, k, want)
print("Edge case: OK")

Edge case: OK


In [22]:
rng = random.Random(2024)
cnt_true = 0
for _ in range(3000):
    n = rng.randint(1, 40)
    p = rng.choice([0.5, 0.2, 0.8])
    s = "".join("1" if rng.random() < p else "0" for _ in range(n))
    k = rng.randint(1, 6)
    want = has_all_codes_brute(s, k)
    cnt_true += want
    check_all(s, k, want)
print("Random 3000 test: OK, số ca True =", cnt_true)

Random 3000 test: OK, số ca True = 843


In [23]:
# Hiệu năng ở ràng buộc lớn: n = 5*10^5
rng = random.Random(99)
s_big = "".join(rng.choice("01") for _ in range(5 * 10**5))
for k in (1, 10, 17, 18, 20):
    t = time.perf_counter()
    r = Solution().hasAllCodes(s_big, k)
    print(f"k={k:2d}: {r!s:5s} {time.perf_counter() - t:.3f}s")
t = time.perf_counter()
r = Solution().hasAllCodes("0" * (5 * 10**5), 10)
print("toàn 0, k=10:", r, f"{time.perf_counter() - t:.3f}s (phải duyệt hết vì không bao giờ đủ)")

k= 1: True  0.000s
k=10: True  0.001s


k=17: False 0.077s
k=18: False 0.088s
k=20: False 0.000s
toàn 0, k=10: False 0.045s (phải duyệt hết vì không bao giờ đủ)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Thiếu cửa sổ cuối: `range(len(s) - k)`

Số cửa sổ là `n − k + 1`, nên phải là `range(len(s) - k + 1)`.

In [24]:
def wrong_missing_last(s, k):
    seen = {s[i:i + k] for i in range(len(s) - k)}      # SAI: thiếu i = n-k
    return len(seen) == 2 ** k

s, k = "01", 1
print("SAI :", wrong_missing_last(s, k), "| đúng:", has_all_codes_bitmask(s, k))
print("Vì range(1) chỉ xét cửa sổ '0', bỏ mất cửa sổ cuối '1'.")

SAI : False | đúng: True
Vì range(1) chỉ xét cửa sổ '0', bỏ mất cửa sổ cuối '1'.


**Sửa:** `range(len(s) - k + 1)` (xem `has_all_codes_set`).

### Lỗi 2 — Quên cắt mask bằng `& full`

Không cắt thì `mask` cứ dài ra, chứa **cả tiền tố** chứ không chỉ `k` ký tự cuối ⇒ các cửa sổ giống nhau bị coi là khác nhau.

In [25]:
def wrong_no_cut(s, k):
    need = 1 << k
    seen = set()
    mask = 0
    for i, c in enumerate(s):
        mask = (mask << 1) | (c == "1")                 # SAI: không & full
        if i >= k - 1:
            seen.add(mask)
    return len(seen) >= need

s, k = "1111", 1
print("SAI :", wrong_no_cut(s, k), "| đúng:", has_all_codes_bitmask(s, k))
print("mask lần lượt là 1, 3, 7, 15 -> 4 'mã khác nhau' dù chỉ có mã '1'.")

SAI : True | đúng: False
mask lần lượt là 1, 3, 7, 15 -> 4 'mã khác nhau' dù chỉ có mã '1'.


**Sửa:** `mask = ((mask << 1) | bit) & ((1 << k) - 1)`.

### Lỗi 3 — Sinh mã bằng `bin(i)[2:]` (thiếu số 0 đầu)

`bin(1)[2:] == "1"`, không phải `"01"`. Chuỗi ngắn hơn dễ "tìm thấy" sai.

In [26]:
def wrong_bin(s, k):
    return all(bin(i)[2:] in s for i in range(2 ** k))  # SAI: '0','1','10','11'

s, k = "0110", 2
print("SAI :", wrong_bin(s, k), "| đúng:", has_all_codes_brute(s, k))
print("Mã '00' bị sinh thành '0', mà '0' có trong s -> trả True sai.")

SAI : True | đúng: False
Mã '00' bị sinh thành '0', mà '0' có trong s -> trả True sai.


**Sửa:** `format(i, f"0{k}b")` hoặc `bin(i)[2:].zfill(k)`.

### Lỗi 4 — Viết `2^k` thay cho `2**k`

Trong Python `^` là XOR: `2 ^ 2 == 0`, `2 ^ 3 == 1`.

In [27]:
def wrong_xor(s, k):
    return len({s[i:i + k] for i in range(len(s) - k + 1)}) == 2 ^ k   # SAI

s, k = "00110110", 2
print("SAI :", wrong_xor(s, k), "| đúng:", has_all_codes_set(s, k), "| 2^k =", 2 ^ k)

SAI : False | đúng: True | 2^k = 0


**Sửa:** `2 ** k` hoặc `1 << k`.

### Lỗi 5 — Ghi nhận mask khi cửa sổ chưa đủ k ký tự

Nếu ghi nhận ngay từ `i = 0`, mask lúc đó chỉ đại diện cho `i+1 < k` ký tự (các bit cao bằng 0 giả) ⇒ có thể "thấy" mã `00…0` không hề có.

In [28]:
def wrong_early(s, k):
    full = (1 << k) - 1
    seen = set()
    mask = 0
    for c in s:
        mask = ((mask << 1) | (c == "1")) & full
        seen.add(mask)                                  # SAI: không kiểm tra i >= k-1
    return len(seen) == 1 << k

s, k = "1100", 2
print("SAI :", wrong_early(s, k), "| đúng:", has_all_codes_bitmask(s, k))
print("Ở i=0 mask = '01' (giả), dù '01' không có trong '1100'.")

SAI : True | đúng: False
Ở i=0 mask = '01' (giả), dù '01' không có trong '1100'.


**Sửa:** chỉ ghi nhận khi `i >= k - 1`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề hỏi "**mọi** chuỗi độ dài k có xuất hiện không / có bao nhiêu chuỗi con phân biệt độ dài k" ⇒ **cửa sổ cố định độ dài k + set**.
- Bảng chữ cái nhỏ (nhị phân, 4 ký tự ADN…) ⇒ mã hóa cửa sổ thành **số nguyên** và trượt bằng dịch bit (2 bit/ký tự với ADN, như bài 187 Repeated DNA Sequences).
- Luôn thử **chuồng bồ câu**: số ứng viên `n − k + 1` so với số cần `2^k`.
- Nhớ công thức trượt: `mask = ((mask << b) | val) & ((1 << (b*k)) - 1)`.
- Cần đếm "mã mới" ⇒ mảng `bytearray(2^k)` + biến đếm nhanh hơn `set` khi `2^k` vừa phải.
- `k ≤ 20` là tín hiệu: `2^k ≈ 10^6` vẫn chứa nổi trong mảng.

## 15. Biến thể của bài

1. **Đếm số mã thiếu** hoặc **liệt kê các mã thiếu** (bài tập 3 bên dưới).
2. **Mã nhỏ nhất bị thiếu**: duyệt `seen` từ 0 lên.
3. **LeetCode 1980 — Find Unique Binary String**: cho `n` chuỗi nhị phân độ dài `n`, tìm một chuỗi không có trong danh sách (đường chéo Cantor).
4. **LeetCode 753 — Cracking the Safe / chuỗi De Bruijn**: chuỗi **ngắn nhất** chứa mọi mã độ dài `k` (độ dài `2^k + k − 1`) — bài tập 5.
5. **LeetCode 187 — Repeated DNA Sequences**: cửa sổ độ dài 10 trên bảng chữ `ACGT`, tìm cửa sổ lặp — cùng kỹ thuật mã hóa 2 bit/ký tự.
6. Bảng chữ cái lớn (26 chữ) hoặc `k` lớn ⇒ dùng **rolling hash đa thức** (Rabin–Karp) với modulo, chấp nhận va chạm hiếm.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | Sinh `2^k` mã, mỗi mã `in s` | `O(2^k·n·k)` | `O(k)` | Kiểm chứng, `k` rất nhỏ |
| 2. Set chuỗi con | Bỏ mọi cửa sổ vào set, so `2^k` | `O(n·k)` | `O(min(n,2^k)·k)` | Phỏng vấn: nhanh gọn, dễ giải thích |
| 3. Set số + dừng sớm | `int(win, 2)`, dừng khi đủ | `O(n·k)` | `O(min(n,2^k))` | Muốn thoát sớm, bước đệm sang bitmask |
| 4. Bitmask trượt | Rolling hash hoàn hảo `O(1)`/bước | `O(n)` | `O(2^k)` | Tối ưu, áp dụng cho bảng chữ nhỏ |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `hasAllCodes("00110", 2)` và `hasAllCodes("1011", 2)` trả gì?

**Bài 2 (tính tay):** Với `s = "1011"`, `k = 2`, liệt kê giá trị `mask` (dạng nhị phân 2 bit) sau mỗi ký tự trong Cách 4, và cho biết những bước nào được ghi nhận.

**Bài 3 (điền TODO):** Viết `missing_codes(s, k)` trả về **danh sách các mã độ dài k không xuất hiện** trong `s`, sắp tăng dần (dạng chuỗi).

**Bài 4 (tự viết lại):** Viết lại `count_distinct_k(s, k)` đếm số chuỗi con **phân biệt** độ dài `k` của chuỗi nhị phân `s` bằng bitmask trượt (không cắt chuỗi).

**Bài 5 (mini challenge):** Viết `de_bruijn(k)` tạo một chuỗi nhị phân **ngắn nhất** chứa mọi mã độ dài `k` (độ dài phải là `2^k + k − 1`).

In [29]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def missing_codes(s: str, k: int) -> List[str]:
    # TODO: thu thập các mask đã thấy, rồi liệt kê i trong range(2**k) chưa thấy
    raise NotImplementedError

# Bài 4 — TODO
def count_distinct_k(s: str, k: int) -> int:
    # TODO: bitmask trượt, chỉ ghi nhận khi i >= k-1
    raise NotImplementedError

# Bài 5 — TODO
def de_bruijn(k: int) -> str:
    # TODO: gợi ý thuật toán "ưu tiên số 1"
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Liệt kê cửa sổ. (2) `"00110"`: `00, 01, 11, 10`. (3) `"1011"` chỉ có 3 cửa sổ < 4.

**Bài 2.** (1) `full = 0b11`. (2) Mỗi bước: dịch trái, OR bit mới, AND `11`. (3) Chỉ ghi nhận từ `i = 1`.

**Bài 3.** (1) Dùng lại vòng lặp Cách 4 với `seen = bytearray(2^k)`. (2) Không dừng sớm. (3) `[format(i, f"0{k}b") for i in range(2**k) if not seen[i]]`.

**Bài 4.** (1) Như Cách 4 nhưng không có chuồng bồ câu. (2) `k > n` ⇒ 0. (3) Trả về `len(set_mask)`.

**Bài 5.** (1) Bắt đầu bằng `"0" * k`. (2) Lặp: thử nối `"1"`, nếu cửa sổ `k` ký tự cuối mới thì giữ; không thì thử `"0"`; nếu cả hai đều cũ thì dừng. (3) Kiểm tra bằng `has_all_codes_bitmask` và độ dài `2^k + k − 1`.

## 19. Đáp án tham khảo cho bài tập

In [30]:
# Bài 1
print("Bài 1:", has_all_codes_bitmask("00110", 2), has_all_codes_bitmask("1011", 2))
assert has_all_codes_bitmask("00110", 2) is True
assert has_all_codes_bitmask("1011", 2) is False

# Bài 2
def masks_of(s, k):
    full = (1 << k) - 1
    mask, out = 0, []
    for i, c in enumerate(s):
        mask = ((mask << 1) | (c == "1")) & full
        out.append((format(mask, f"0{k}b"), i >= k - 1))
    return out
print("Bài 2:", masks_of("1011", 2))
assert masks_of("1011", 2) == [("01", False), ("10", True), ("01", True), ("11", True)]

Bài 1: True False
Bài 2: [('01', False), ('10', True), ('01', True), ('11', True)]


In [31]:
def missing_codes_ans(s: str, k: int) -> List[str]:
    need = 1 << k
    full = need - 1
    seen = bytearray(need)
    mask = 0
    for i, c in enumerate(s):
        mask = ((mask << 1) | (c == "1")) & full
        if i >= k - 1:
            seen[mask] = 1
    return [format(i, f"0{k}b") for i in range(need) if not seen[i]]

def count_distinct_k_ans(s: str, k: int) -> int:
    if k > len(s):
        return 0
    full = (1 << k) - 1
    seen = set()
    mask = 0
    for i, c in enumerate(s):
        mask = ((mask << 1) | (c == "1")) & full
        if i >= k - 1:
            seen.add(mask)
    return len(seen)

assert missing_codes_ans("0110", 2) == ["00"]
assert missing_codes_ans("00110110", 2) == []
assert count_distinct_k_ans("00110110", 2) == 4
rng = random.Random(5)
for _ in range(500):
    s = "".join(rng.choice("01") for _ in range(rng.randint(1, 30)))
    k = rng.randint(1, 5)
    wins = {s[i:i + k] for i in range(len(s) - k + 1)}
    assert count_distinct_k_ans(s, k) == len(wins)
    assert missing_codes_ans(s, k) == [c for c in (format(i, f"0{k}b") for i in range(1 << k)) if c not in wins]
    assert (missing_codes_ans(s, k) == []) == has_all_codes_bitmask(s, k)
print("Bài 3, 4: OK", missing_codes_ans("0110", 2))

Bài 3, 4: OK ['00']


In [32]:
def de_bruijn_ans(k: int) -> str:
    seq = ["0"] * k
    seen = {"0" * k}
    while True:
        for b in "10":                       # ưu tiên '1'
            cand = "".join(seq[len(seq) - k + 1:]) + b if k > 1 else b
            if cand not in seen:
                seen.add(cand)
                seq.append(b)
                break
        else:
            break
    return "".join(seq)

for k in range(1, 11):
    d = de_bruijn_ans(k)
    assert len(d) == (1 << k) + k - 1, (k, len(d))
    assert has_all_codes_bitmask(d, k)
print("Bài 5: de_bruijn(3) =", de_bruijn_ans(3), "| OK với k = 1..10")

Bài 5: de_bruijn(3) = 0001110100 | OK với k = 1..10


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template cửa sổ bitmask độ dài k:**

```python
need = 1 << k
if len(s) - k + 1 < need:          # chuồng bồ câu
    return False
full, seen, mask, cnt = need - 1, bytearray(need), 0, 0
for i, c in enumerate(s):
    mask = ((mask << 1) | (c == "1")) & full
    if i >= k - 1 and not seen[mask]:
        seen[mask] = 1; cnt += 1
        if cnt == need: return True
return False
```

**Checklist:**
- [ ] Dùng `1 << k` hoặc `2 ** k`, **không** dùng `2 ^ k`.
- [ ] Số cửa sổ là `n − k + 1`; đã xử lý `k > n`.
- [ ] Có `& full` khi trượt mask.
- [ ] Chỉ ghi nhận khi `i >= k − 1`.
- [ ] Nếu sinh mã dạng chuỗi: đệm số 0 (`format(i, f"0{k}b")`).
- [ ] Đã thử `s` toàn 0, `k = 1`, `k = len(s)`, chuỗi De Bruijn.